# Fine-Tune 4 Expert (`Qwen/Qwen3-1.7B`, 6.000 Sampel) -> Merge `Qwen3 MoE` (~5B-A2.3B) -> Export GGUF (`Q4_K_M`) di Colab T4

**Fitur Utama Notebook Ini:**
- Mendukung **Google Drive persistence** (opsional) agar hasil training tiap expert tidak hilang jika Colab disconnect.
- Pembersihan disk otomatis (menghapus `F16.gguf` setelah quantize ke `Q4_K_M`) agar aman di batas disk Colab.
- Dilengkapi **8 `positive_prompts` per expert** untuk routing `cheap_embed` yang tajam.
- Dilengkapi **Plan B (Cell 6):** Konversi 4 LoRA ke GGUF LoRA Adapters (`llama-server --lora`) jika suatu saat konversi MoE -> GGUF bermasalah.

**Prasyarat:** Pilih **Runtime -> Change runtime type -> T4 GPU** sebelum menjalankan.

In [ ]:
!nvidia-smi
!df -h /content
!pip install -q unsloth trl peft accelerate bitsandbytes datasets mergekit pyyaml
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -B /content/llama.cpp/build /content/llama.cpp && cmake --build /content/llama.cpp/build --target llama-quantize llama-cli -j4

In [ ]:
import os
from google.colab import files

USE_DRIVE = False  # Ubah ke True jika ingin auto-backup hasil ke Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/audit_moe_backup'
    os.makedirs(BACKUP_DIR, exist_ok=True)
else:
    BACKUP_DIR = None

os.makedirs('/content/datasets', exist_ok=True)
print('Upload 4 file .jsonl (expert_1_accounting, expert_2_audit, expert_3_tax_id, expert_4_agentic) + opsional eval_heldout_200.jsonl:')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'/content/datasets/{fname}', 'wb') as f:
        f.write(data)
print('File tersimpan:', sorted(os.listdir('/content/datasets')))

In [ ]:
import gc, shutil, torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

EXPERTS = [
    'expert_1_accounting',
    'expert_2_audit',
    'expert_3_tax_id',
    'expert_4_agentic',
]

for expert_name in EXPERTS:
    merged_out = f'/content/experts/{expert_name}'
    lora_out = f'/content/loras/{expert_name}'
    if os.path.exists(merged_out):
        print(f'[SKIP] {expert_name} sudah ada di {merged_out}')
        continue

    print(f'\n=== TRAINING {expert_name} ===')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen3-1.7B-unsloth-bnb-4bit',
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=2026,
    )

    ds = load_dataset('json', data_files=f'/content/datasets/{expert_name}.jsonl', split='train')
    def format_chat(batch):
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch['messages']]
        return {'text': texts}
    ds = ds.map(format_chat, batched=True)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field='text',
        max_seq_length=2048,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=15,
            num_train_epochs=2,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=20,
            output_dir=f'/content/outputs/{expert_name}',
            save_strategy='no',
        ),
    )
    trainer.train()

    model.save_pretrained(lora_out)
    model.save_pretrained_merged(merged_out, tokenizer, save_method='merged_16bit')
    shutil.rmtree(f'/content/outputs/{expert_name}', ignore_errors=True)

    if BACKUP_DIR:
        shutil.copytree(lora_out, f'{BACKUP_DIR}/loras/{expert_name}', dirs_exist_ok=True)

    del model, trainer, ds
    gc.collect()
    torch.cuda.empty_cache()
print('Semua 4 expert selesai dilatih!')

In [ ]:
import os, yaml

moe_cfg = {
    'base_model': '/content/experts/expert_1_accounting',
    'gate_mode': 'cheap_embed',
    'lazy_unpickle': True,
    'dtype': 'float16',
    'experts_per_token': 2,
    'experts': [
        {
            'source_model': '/content/experts/expert_1_accounting',
            'positive_prompts': [
                'Buat jurnal umum dan jurnal penyesuaian sesuai SAK, SAK-EP, atau PSAP BUMD.',
                'Petakan setoran LPP loket dan rekap DRD tagihan air ke Chart of Accounts (CoA).',
                'Buat jurnal majemuk pembayaran jasa konsultan termasuk PPN Masukan dan potongan PPh 23.',
                'Catat jurnal pembelian persediaan bahan kimia secara kredit beserta PPN.',
                'Susun jurnal penyusutan aset tetap, penyisihan piutang tak tertagih, dan beban akrual.',
                'Buat jurnal koreksi reklasifikasi belanja modal CapEx dari beban pemeliharaan.',
                'Catat penerimaan uang muka sambungan baru sebagai pendapatan diterima dimuka.',
                'Susun Neraca Lajur, Buku Besar, Laba Rugi, Neraca, dan Arus Kas.',
            ],
        },
        {
            'source_model': '/content/experts/expert_2_audit',
            'positive_prompts': [
                'Hitung EBITDA, EBITDA Margin, dan prosedur audit substantif aset tetap.',
                'Susun analisis varians RKAP pendapatan dan beban operasional.',
                'Buat proyeksi keuangan proforma tahun depan berdasarkan asumsi pertumbuhan.',
                'Lakukan rekonsiliasi bank antara saldo rekening koran, DIT, dan cek beredar.',
                'Hitung Current Ratio, Debt to Equity Ratio (DER), dan DSCR.',
                'Evaluasi rasio efisiensi penagihan piutang air BUMD berdasarkan DRD.',
                'Tetapkan Overall Materiality, Performance Materiality, dan Clearly Trivial sesuai SA 320.',
                'Buat Kertas Kerja Audit (KKA), sampling uji substantif, dan audit trail.',
            ],
        },
        {
            'source_model': '/content/experts/expert_3_tax_id',
            'positive_prompts': [
                'Hitung PPN non-mewah tarif 12% atas DPP Nilai Lain 11/12 sesuai PMK 131/2024.',
                'Hitung PPN 12% penuh untuk penyerahan barang tergolong mewah.',
                'Tentukan tarif dan pemotongan PPh Pasal 23 jasa/royalti ber-NPWP maupun tanpa NPWP.',
                'Hitung PPh Pasal 4 ayat 2 Final sewa tanah/bangunan 10% dan UMKM 0,5%.',
                'Hitung pemungutan PPh Pasal 22 pengadaan barang 1,5%.',
                'Hitung PPh Pasal 21 bulanan skema TER kategori A, B, dan C sesuai PP 58/2023.',
                'Susun rekonsiliasi koreksi fiskal positif dan negatif untuk PPh Badan 22%.',
                'Lakukan ekualisasi omzet SPT PPh Badan terhadap DPP SPT Masa PPN.',
            ],
        },
        {
            'source_model': '/content/experts/expert_4_agentic',
            'positive_prompts': [
                'Scan daftar file keuangan klien (.xls, .xlsx, .docx, .pdf) dalam workspace.',
                'Inspeksi struktur sheet, header, dan sel berformula pada file template Excel klien.',
                'Tulis jurnal ke file Excel output baru tanpa menimpa file asli dan tanpa merusak rumus.',
                'Baca kontrak DOCX dan jalankan OCR lokal pada hasil scan PDF faktur pajak.',
                'Jalankan query SQLite3 read-only SELECT untuk analitik saldo dan sampling audit.',
                'Verifikasi balance gate Debit == Kredit dan Aset == Kewajiban + Ekuitas.',
                'Bangun dashboard interaktif non-HTML pada Excel Native Chart atau aplikasi desktop GUI.',
                'Lakukan operasi RWED file aman termasuk patch edit dan blokir delete tanpa konfirmasi.',
            ],
        },
    ],
}
with open('/content/mergekit_moe_3b.yaml', 'w') as f:
    yaml.safe_dump(moe_cfg, f, allow_unicode=True)

!mergekit-moe /content/mergekit_moe_3b.yaml /content/audit_moe_merged --copy-tokenizer --allow-crimes && \
echo 'Merge MoE selesai, lanjut ke Router Calibration di cell berikutnya!'

In [ ]:
# Router Calibration: bekukan seluruh bobot expert, latih HANYA layer router (mlp.gate) ~229K params
# agar pemilihan expert akurat dan skema JSON tidak tercampur!
import json, random, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling

calib_samples = []
for exp in ['expert_1_accounting', 'expert_2_audit', 'expert_3_tax_id', 'expert_4_agentic']:
    with open(f'/content/datasets/{exp}.jsonl', 'r', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    calib_samples.extend(rows[:150])  # 150 x 4 = 600 sampel seimbang
random.Random(2026).shuffle(calib_samples)

tok = AutoTokenizer.from_pretrained('/content/audit_moe_merged')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def tokenize_fn(row):
    txt = tok.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
    enc = tok(txt, truncation=True, max_length=384)
    enc['labels'] = enc['input_ids'].copy()
    return enc

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
calib_ds = Dataset.from_list(calib_samples).map(tokenize_fn, remove_columns=['messages'])

moe_model = AutoModelForCausalLM.from_pretrained(
    '/content/audit_moe_merged',
    torch_dtype=torch.float16,
    device_map='auto',
    max_memory={0: '11GiB', 'cpu': '12GiB'},
)
moe_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
moe_model.enable_input_require_grads()

trainable_params = 0
for name, param in moe_model.named_parameters():
    if name.endswith('mlp.gate.weight'):
        param.data = param.data.float()
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        param.requires_grad = False
print(f'Router Calibration trainable params: {trainable_params:,}')

router_trainer = Trainer(
    model=moe_model,
    args=TrainingArguments(
        output_dir='/content/router_calib_out',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=5e-4,
        fp16=True,
        logging_steps=10,
        save_strategy='no',
    ),
    train_dataset=calib_ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)
router_trainer.train()
for name, param in moe_model.named_parameters():
    if name.endswith('mlp.gate.weight'):
        param.data = param.data.half()
moe_model.save_pretrained('/content/audit_moe_calibrated', safe_serialization=False, max_shard_size='2GB')
tok.save_pretrained('/content/audit_moe_calibrated')

del moe_model, router_trainer, calib_ds
import gc; gc.collect(); torch.cuda.empty_cache()

!python3 /content/llama.cpp/convert_hf_to_gguf.py /content/audit_moe_calibrated --outfile /content/asadeltek-indo-tax-moe-5b-f16.gguf --outtype f16 && \
/content/llama.cpp/build/bin/llama-quantize /content/asadeltek-indo-tax-moe-5b-f16.gguf /content/asadeltek-indo-tax-moe-5b.gguf Q4_K_M && \
rm -f /content/asadeltek-indo-tax-moe-5b-f16.gguf && \
/content/llama.cpp/build/bin/llama-cli -m /content/asadeltek-indo-tax-moe-5b.gguf -p "Hitung PPN non-mewah PMK 131/2024 atas DPP Rp 100.000.000 dalam JSON:" -n 160 --single-turn

In [ ]:
# PLAN B (Opsional): Jalankan cell ini HANYA jika konversi MoE -> GGUF gagal.
# Mengonversi 4 LoRA adapter ke GGUF LoRA agar bisa dipakai hot-swap di llama.cpp (--lora) dengan RAM ~1.5 GB!
RUN_PLAN_B_LORA_GGUF = False
if RUN_PLAN_B_LORA_GGUF:
    os.makedirs('/content/gguf_loras', exist_ok=True)
    for exp in ['expert_1_accounting', 'expert_2_audit', 'expert_3_tax_id', 'expert_4_agentic']:
        !python3 /content/llama.cpp/convert_lora_to_gguf.py /content/loras/{exp} --base Qwen/Qwen3-1.7B --outfile /content/gguf_loras/{exp}-lora.gguf --outtype f16
    print('Plan B GGUF LoRA adapters siap di /content/gguf_loras/')

In [ ]:
# Upload langsung ke Hugging Face (otomatis buat repo baru dengan aman via Colab Secrets / login)
from huggingface_hub import HfApi, notebook_login
import os

HF_REPO_ID = 'USERNAME_HF_KAMU/asadeltek-indo-tax-moe-5b'  # Ganti USERNAME_HF_KAMU
HF_PRIVATE = True  # Ubah ke False jika ingin repo publik

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    notebook_login()
    api = HfApi()
else:
    api = HfApi(token=hf_token)

api.create_repo(repo_id=HF_REPO_ID, repo_type='model', private=HF_PRIVATE, exist_ok=True)

gguf_file = '/content/asadeltek-indo-tax-moe-5b.gguf'
if os.path.exists(gguf_file):
    api.upload_file(
        path_or_fileobj=gguf_file,
        path_in_repo='asadeltek-indo-tax-moe-5b.gguf',
        repo_id=HF_REPO_ID,
        repo_type='model',
    )
    api.upload_file(
        path_or_fileobj='/content/mergekit_moe_3b.yaml',
        path_in_repo='mergekit_moe_3b.yaml',
        repo_id=HF_REPO_ID,
        repo_type='model',
    )
    print(f'Berhasil upload ke https://huggingface.co/{HF_REPO_ID}')

In [ ]:
import shutil
from google.colab import files

target_gguf = '/content/asadeltek-indo-tax-moe-5b.gguf'
if BACKUP_DIR and os.path.exists(target_gguf):
    shutil.copy2(target_gguf, f'{BACKUP_DIR}/asadeltek-indo-tax-moe-5b.gguf')
    print('Berhasil backup GGUF ke Google Drive:', BACKUP_DIR)
elif os.path.exists(target_gguf):
    files.download(target_gguf)